# Walmart Price Prediction
## CS1783 Artificial Intelligence Programming
## Jacob Shultz

### Assignment:

"Develop and train a Long Short-Term Memory (LSTM) neural network using PyTorch to predict future Walmart sales based on historical sales data.

The model must predict 28 days of future sales rather than simply predicting the next day.

Aggregate daily sales for a single Walmart store.

Requirements

1. Analyze, clean, and preprocess the dataset.
2. Design, implement, and train an LSTM using PyTorch.
3. Predict daily sales for the next 28 days.
4. Evaluate predictions against actual sales.
5. Experiment with input features, sequence lengths, and model hyperparameters.
6. Compare the LSTM against a simple historical forecasting method.

Be sure to document and explain variations in your network design and hyper parameters. Submit a techinal paper along with your source code."

## 0. Global

In [116]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
from sklearn.preprocessing import MinMaxScaler

In [117]:
# Import the CSV files
df_calendar = pd.read_csv('data/calendar.csv')  # Dates on which products are sold
df_sales = pd.read_csv('data/sales_train_evaluation.csv')  # Sales data for training set
# df_sales_train_validation = pd.read_csv('data/sales_train_validation.csv')  # Sales data for validation set, instructed to discard for this assignment.
# df_sample_submission = pd.read_csv('data/sample_submission.csv')  # example competition submission, irrelevant
# df_sell_prices = pd.read_csv('data/sell_prices.csv')    # Prices of the products sold per store and date. Irrelevant, we're not predicting prices.

In [118]:
df_calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [119]:
print(df_sales.head())
print(df_sales.shape)

                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_evaluation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_evaluation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_evaluation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_evaluation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id  d_1  d_2  d_3  d_4  ...  d_1932  d_1933  d_1934  d_1935  d_1936  \
0       CA    0    0    0    0  ...       2       4       0       0       0   
1       CA    0    0    0    0  ...       0       1       2       1       1   
2       CA    0    0    0    0  ...       1       0       2       0       0   
3       CA    0    0    0    0  ...       1       1       0       4       0   
4       CA    0    0    0    0  ...       0       0       0       2       1   

   d_1937  d_1938  d_1939  d_1940  d_1941  


## 1. Data Processing and Analysis

In [120]:
# sales data contains data for multiple stores.
df_sales['store_id'].unique()

<StringArray>
['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2',
 'WI_3']
Length: 10, dtype: str

In [121]:
# slim data down to the first store, CA_1.
df_sales = df_sales[df_sales['store_id'] == 'CA_1']
print(df_sales['store_id'].unique().tolist())
print(df_sales.head())
print(df_sales.shape)

['CA_1']
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_evaluation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_evaluation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_evaluation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_evaluation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id  d_1  d_2  d_3  d_4  ...  d_1932  d_1933  d_1934  d_1935  d_1936  \
0       CA    0    0    0    0  ...       2       4       0       0       0   
1       CA    0    0    0    0  ...       0       1       2       1       1   
2       CA    0    0    0    0  ...       1       0       2       0       0   
3       CA    0    0    0    0  ...       1       1       0       4       0   
4       CA    0    0    0    0  ...       0       0       0       2       1   

   d_1937  d_1938  d_1939  d_1940  

In [122]:
# collapse data for each item into one daily total sales value
df_daily_sales = df_sales.iloc[:, 6:]
print(df_daily_sales.head())
print('--------------------------------------')

# sum each column
total_daily_sales = df_daily_sales.sum(numeric_only=True)   # as a series
print(total_daily_sales.head())
print(total_daily_sales.shape)
print('--------------------------------------')

   d_1  d_2  d_3  d_4  d_5  d_6  d_7  d_8  d_9  d_10  ...  d_1932  d_1933  \
0    0    0    0    0    0    0    0    0    0     0  ...       2       4   
1    0    0    0    0    0    0    0    0    0     0  ...       0       1   
2    0    0    0    0    0    0    0    0    0     0  ...       1       0   
3    0    0    0    0    0    0    0    0    0     0  ...       1       1   
4    0    0    0    0    0    0    0    0    0     0  ...       0       0   

   d_1934  d_1935  d_1936  d_1937  d_1938  d_1939  d_1940  d_1941  
0       0       0       0       0       3       3       0       1  
1       2       1       1       0       0       0       0       0  
2       2       0       0       0       2       3       0       1  
3       0       4       0       1       3       0       2       6  
4       0       2       1       0       0       2       1       0  

[5 rows x 1941 columns]
--------------------------------------
d_1    4337
d_2    4155
d_3    2816
d_4    3051
d_5    2630
dtype

In [123]:
# attach the new series to the calendar data
df_calendar['sales'] = df_calendar['d'].map(total_daily_sales)
df_calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sales
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,4337.0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0,4155.0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0,2816.0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0,3051.0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1,2630.0


In [ ]:
# drop less useful columns
df_master = df_calendar.drop(columns=['snap_TX', 'snap_WI', 'wday', 'wm_yr_wk', 'date', 'event_name_1', 'event_name_2'])

# deal with missing values
df_master = df_master.dropna(subset='sales')

# remove outliers
m = df_master['sales'].mean()
std = df_master['sales'].std()
lb = m - 3 * std
ub = m + 3 * std
df_master = df_master[(df_master['sales'] >= lb) & (df_master['sales'] <= ub)]

# normalize
scaler = MinMaxScaler()
df_master['sales'] = scaler.fit_transform(df_master[['sales']]) # note: will have to undo this later. how...

df_master.head()

,weekday,month,year,d,event_type_1,event_type_2,snap_CA,sales
0,Saturday,1,2011,d_1,NaN,NaN,0,0.478530
1,Sunday,1,2011,d_2,NaN,NaN,0,0.442181
2,Monday,1,2011,d_3,NaN,NaN,0,0.174755
3,Tuesday,2,2011,d_4,NaN,NaN,1,0.221690
4,Wednesday,2,2011,d_5,NaN,NaN,1,0.137607
